# 03 · Đánh giá

Điểm của cuộc thi là **PA (penalized accuracy)**: câu đúng +1, câu sai −1, bỏ trống 0, chia cho số câu.

1. **Một lượt chạy có nhãn** (train / holdout): điểm, theo loại / shop / dạng câu, nhóm lỗi, ngưỡng bỏ phiếu và oracle@K.
2. **So hai adapter trên cùng câu hỏi** (ví dụ adapter BF16 mới với `sql_rft_v3` trên holdout): chênh lệch PA, chuyển đổi đúng / sai / bỏ trống,
   khoảng tin cậy bootstrap theo shop.
3. **Các bản nộp public:** so với nhau, không cần nhãn (điểm public chỉ có ở mức tổng, không bao giờ làm nhãn cho từng câu).

In [ ]:
# Setup: tìm checkout của repo này, nạp datamate.notebook, phát hiện dữ liệu, GPU, thư mục model và runs.
#   local   mở notebook trong notebooks/ của checkout (hoặc đặt biến môi trường DATAMATE_REPO), không cần cài thêm.
#   Kaggle  đính kèm bản sao checkout làm dataset (Add data) cùng dữ liệu BTC (dataset riêng tư), bật Internet để cài gói.
#   Colab   chép checkout vào MyDrive/datamate/code, dữ liệu BTC vào MyDrive/datamate/data.
# Kaggle / Colab: code được chép sang ổ ghi được (input của Kaggle chỉ đọc, Drive chậm); kết quả ghi vào env.runs, được giữ sau phiên.
import os, shutil, sys
from pathlib import Path

MARKER = "datamate/notebook.py"


def find_checkout() -> Path:
    if os.environ.get("DATAMATE_REPO"):
        return Path(os.environ["DATAMATE_REPO"]).resolve()
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / MARKER).exists():
            return c.resolve()
    if Path("/kaggle/input").exists():
        found, work = sorted(p.parents[1] for p in Path("/kaggle/input").glob(f"**/{MARKER}")), Path("/tmp/datamate")
    elif "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        found, work = [p for p in [Path("/content/drive/MyDrive/datamate/code")] if (p / MARKER).exists()], Path("/content/datamate")
    else:
        found, work = [], None
    if not found:
        raise FileNotFoundError("không tìm thấy checkout datamate: mở notebook trong notebooks/ của repo, hoặc đặt DATAMATE_REPO")
    if not (work / MARKER).exists():
        shutil.copytree(found[0], work, ignore=shutil.ignore_patterns(".git", "runs", "models", ".venv*", "__pycache__"))
    return work


root = find_checkout()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("base")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
RUN = "runs/nb_train_qwen35_sft_a6000"    # lượt chạy có nhãn (thư mục có trace.jsonl hoặc shard*/trace.jsonl), ví dụ adapter mới trên holdout
BASELINE = "runs/nb_train_sql_rft_v3"     # lượt chạy để so trên cùng câu hỏi (None = bỏ qua phần 2)
SUBS = {"v6 (0.63)": "runs/sub_v6", "v8 (0.639)": "runs/sub_v8", "v9 (0.683)": "runs/sub_v9"}   # bản nộp public cần so

In [ ]:
from datamate import scoring, voting
from datamate.hints import DEBT_WORDS
from datamate.ingest import norm


def load_trace(run):
    run = env.path(run)
    files = [run / "trace.jsonl"] if (run / "trace.jsonl").exists() else sorted(run.glob("shard*/trace.jsonl"))
    return {r["id"]: r for p in files for r in read(p)}


gold = {a["id"]: a for a in read(env.split("train") / "answers.jsonl")}
questions = {q["id"]: q for q in read(env.data / "train/questions.jsonl")}
trace = load_trace(RUN)
print(len(trace), "câu trong", env.path(RUN), "" if trace else "=> chạy 02_inference với SPLIT = 'train' trước")

## 1. Điểm của lượt chạy

- **PA:** điểm của cuộc thi. **precision:** tỷ lệ đúng trong các câu có trả lời. **coverage:** tỷ lệ câu có trả lời.
- **Khoảng tin cậy:** lấy mẫu lại theo shop (bootstrap), cho biết điểm dao động bao nhiêu nếu đổi sang tập shop khác.

In [ ]:
if trace:
    report, rows = scoring.evaluate({i: gold[i] for i in trace}, {i: r["final"] for i, r in trace.items()}, questions)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in report.items() if k in ("N", "correct", "wrong", "abstain", "PA", "precision", "coverage")})
    print("PA 95% CI (shop bootstrap):", [round(x, 3) for x in report["PA_cluster_bootstrap_95"]])
    display(pd.DataFrame(report["breakdowns"]["gold_type"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("N", ascending=False))

In [ ]:
if trace:
    by_shop = pd.DataFrame(report["breakdowns"]["business_id"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("PA")
    by_shop.plot.barh(y="PA", figsize=(6, max(2, 0.25 * len(by_shop))), legend=False, title="PA by shop")
    display(by_shop.head(8))

### Nhóm lỗi

Mỗi câu chưa đúng thuộc một nhóm: không mẫu nào đúng (cần model / dữ liệu SFT tốt hơn), có mẫu đúng nhưng nhóm sai thắng phiếu, hoặc nhóm đúng
đứng đầu nhưng dưới ngưỡng nên bỏ trống (cần bỏ phiếu / ngưỡng tốt hơn). Fine-tuning tác động chủ yếu lên nhóm đầu.

In [ ]:
def kind(r):
    g = gold[r["id"]]
    s = scoring.score(r["final"], g)
    if s == 1: return "right"
    if not any(p and scoring.score(p, g) == 1 for p in (x["pred"] for x in r["samples"])): return "no right sample"
    if s == -1: return "wrong: a wrong group won"
    top_right = bool(r.get("vote")) and scoring.score(r["vote"], g) == 1
    return "abstain: right group on top, below threshold" if top_right else "abstain: right sample, another group on top"


pd.Series(Counter(kind(r) for r in trace.values())).sort_values(ascending=False).to_frame("questions")

### Ngưỡng bỏ phiếu và oracle@K

- **Ngưỡng bỏ phiếu:** tỷ lệ mẫu đồng thuận tối thiểu để trả lời. Ngưỡng cao thì ít sai hơn nhưng bỏ trống nhiều hơn. Bảng bỏ phiếu lại trên các
  mẫu đã lưu với từng ngưỡng.
- **oracle@K:** tỷ lệ câu có ít nhất một mẫu đúng trong K mẫu, tức mức điểm tối đa nếu luôn chọn được mẫu đúng.

In [ ]:
def replay(r, tau):
    texts, preds = [s["text"] for s in r["samples"]], [s["pred"] for s in r["samples"]]
    pool = voting.debt_pool(texts, 8, bool(DEBT_WORDS.search(norm(r["question"]))))
    ans, conf = voting.vote([preds[j] for j in pool], len(pool))
    return voting.final_answer(ans, conf, tau, r["final"].get("reason") == "period outside the shop's data dates", True)


if trace:
    sweep = pd.Series({t: sum(scoring.score(replay(r, t), gold[i]) for i, r in trace.items()) / len(trace) for t in (0.25, 0.375, 0.5, 0.625, 0.75)}, name="PA")
    oracle = sum(any(p and scoring.score(p, gold[i]) == 1 for p in (s["pred"] for s in r["samples"])) for i, r in trace.items()) / len(trace)
    print(f"oracle (any sample right): {oracle:.3f}")
    display(sweep.to_frame())

### Theo dạng câu hỏi

In [ ]:
from datamate import hints as H

shape = lambda q: ("debt" if DEBT_WORDS.search(norm(q)) else "profit" if H.PROFIT.search(norm(q)) else "trips" if H.TRIPS.search(norm(q))
                   else "lots" if H.LOT.search(norm(q)) else "other")
df = pd.DataFrame([{"shape": shape(r["question"]), "score": scoring.score(r["final"], gold[i])} for i, r in trace.items()])
df.groupby("shape")["score"].agg(n="size", PA="mean").sort_values("n", ascending=False).round(3) if len(df) else None

## 2. So hai adapter trên cùng câu hỏi

Chỉ tính các câu có ở cả hai lượt chạy. **Chênh lệch PA** = PA(`RUN`) − PA(`BASELINE`); khoảng tin cậy 95% lấy mẫu lại theo shop (các câu của
một shop đi cùng nhau), nên phản ánh dao động khi đổi sang shop khác. Bảng chuyển đổi cho biết bao nhiêu câu đổi từ đúng sang sai và ngược lại.

Lưu ý khi đọc trên holdout: `sql_rft_v3` đã học 390 câu holdout nên điểm của nó bị thổi phồng; adapter train trên dữ liệu không có shop holdout
(`04_training_data`, `EXCLUDE_HOLDOUT = True`) mà vẫn ngang hoặc hơn v3 là bằng chứng mạnh.

In [ ]:
import random

base = load_trace(BASELINE) if BASELINE else {}
common = sorted(set(trace) & set(base))
if common:
    new = {i: scoring.score(trace[i]["final"], gold[i]) for i in common}
    old = {i: scoring.score(base[i]["final"], gold[i]) for i in common}
    shop = {i: questions[i]["business_id"] for i in common}
    shops = sorted(set(shop.values()))
    per_shop = pd.DataFrame([{"shop": s, "N": sum(shop[i] == s for i in common), "PA baseline": sum(old[i] for i in common if shop[i] == s),
                              "PA run": sum(new[i] for i in common if shop[i] == s)} for s in shops]).set_index("shop")
    per_shop[["PA baseline", "PA run"]] = per_shop[["PA baseline", "PA run"]].div(per_shop["N"], axis=0)
    per_shop["delta"] = per_shop["PA run"] - per_shop["PA baseline"]
    delta = (sum(new.values()) - sum(old.values())) / len(common)
    rng, boot = random.Random(0), []
    by = {s: [i for i in common if shop[i] == s] for s in shops}
    for _ in range(2000):
        pick = [i for s in rng.choices(shops, k=len(shops)) for i in by[s]]
        boot.append((sum(new[i] for i in pick) - sum(old[i] for i in pick)) / len(pick))
    boot.sort()
    print(f"{len(common)} câu chung, {len(shops)} shop | PA baseline {sum(old.values()) / len(common):.4f} -> run {sum(new.values()) / len(common):.4f} "
          f"| delta {delta:+.4f}, 95% CI [{boot[50]:+.4f}, {boot[1949]:+.4f}]")
    name = {1: "right", 0: "abstain", -1: "wrong"}
    display(pd.crosstab(pd.Series([name[old[i]] for i in common], name="baseline"), pd.Series([name[new[i]] for i in common], name="run")))
    display(per_shop.round(3).sort_values("delta"))
else:
    print("không có câu chung giữa RUN và BASELINE (hoặc BASELINE = None)")

## 3. So các bản nộp public (không nhãn)

Số câu trả lời, mức trùng đáp án giữa các file, và chỗ khác nhau theo shop giữa hai bản cuối.

In [ ]:
def key(a):
    t, x = a.get("type"), a.get("answer")
    if t == "abstain" or (x is None and t != "unanswerable"): return "ABSTAIN"
    return "UNANSWERABLE" if t == "unanswerable" else json.dumps([t, round(x, 2) if isinstance(x, float) else x], ensure_ascii=False, sort_keys=True)


pq = {q["id"]: q for q in read(env.split("public_test") / "questions.jsonl")}
subs = {name: {r["id"]: key(r) for r in read(env.path(p) / "submission.jsonl")} for name, p in SUBS.items() if (env.path(p) / "submission.jsonl").exists()}
print("bản nộp tìm thấy:", list(subs) or "không có")
pd.DataFrame({n: {"answered": sum(v != "ABSTAIN" for v in s.values()), "unanswerable": sum(v == "UNANSWERABLE" for v in s.values())} for n, s in subs.items()}).T

In [ ]:
names = list(subs)
pd.DataFrame([[sum(subs[a].get(i) == subs[b].get(i) for i in pq) / len(pq) for b in names] for a in names], index=names, columns=names).style.format("{:.1%}")

In [ ]:
if len(names) >= 2:
    a, b = names[-2], names[-1]
    display(pd.DataFrame([{"shop": q["business_id"], "changed": subs[a].get(i) != subs[b].get(i), "newly answered": subs[a].get(i) == "ABSTAIN" != subs[b].get(i),
                           "newly abstained": subs[b].get(i) == "ABSTAIN" != subs[a].get(i)} for i, q in pq.items()]).groupby("shop").sum())